# 02c — CNN from Scratch Training Pipeline

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** Custom 4-block CNN (~500K params, no pretrained weights)

- Input size: 224x224, normalization: [0, 1]
- Single training run (no freeze/unfreeze stages)
- Purpose: Deep learning baseline to quantify transfer learning benefit

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import torch
import torch.nn as nn

import config
from src.dataset import get_dataloaders, compute_class_weights
from src.model_cnn import CrackCNN
from src.trainer import train_model
from src.evaluation import plot_training_history, evaluate_model
from src.device import print_device_summary, get_device, set_seed

# Reproducibility
set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "cnn")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
NUM_WORKERS = device_config["num_workers"]
BATCH_SIZE = config.CNN_BATCH_SIZE
if not device_config["gpu_detected"]:
    BATCH_SIZE = min(BATCH_SIZE, 16)

print(f"\nModel: CNN from Scratch")
print(f"Image size: {config.CNN_IMG_SIZE}, Batch: {BATCH_SIZE}")
print(f"Device: {device}")

In [ ]:
# 224x224 with [0,1] normalization
train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=BATCH_SIZE,
    img_size=config.CNN_IMG_SIZE,
    normalize="rescale",
    num_workers=NUM_WORKERS,
)

class_weight_dict = compute_class_weights()
class_weight_tensor = torch.tensor(
    [class_weight_dict[i] for i in range(config.NUM_CLASSES)], dtype=torch.float32,
).to(device)

In [ ]:
model = CrackCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(model)

## Train

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weight_tensor,
    label_smoothing=config.CNN_LABEL_SMOOTHING,
)
optimizer = torch.optim.Adam(model.parameters(), lr=config.CNN_LR)

history = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.CNN_EPOCHS,
    output_dir=OUTPUT_DIR,
    stage=1,
    patience=config.CNN_EARLY_STOPPING_PATIENCE,
    model_name="cnn",
)

## Training Curves & Evaluation

In [ ]:
plot_training_history(
    [history], output_dir=OUTPUT_DIR,
    stage_names=["Training"], model_name="cnn",
)

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "models", "best_model.pt"))

_, _, test_loader = get_dataloaders(
    batch_size=BATCH_SIZE, img_size=config.CNN_IMG_SIZE,
    normalize="rescale", num_workers=NUM_WORKERS,
)
metrics = evaluate_model(model, test_loader, device, output_dir=OUTPUT_DIR, model_name="cnn")